In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",  # axes with grid
    rc={
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
    },
)

Load data

In [ ]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


logprob_results_path = "../analysis/results/logprob_acc_merged.json"
logprob_results = pd.read_json(logprob_results_path, orient="records")

logprob_norm_results_path = "../analysis/results/logprob_acc_norm_merged.json"
logprob_norm_results = pd.read_json(logprob_norm_results_path, orient="records")

generative_results_path = "../analysis/generative_tails.json"
generative_results = pd.read_json(generative_results_path, orient="records")

In [ ]:
raw_results_path = "../logs/silent-norm-final-v1/results.json"
raw_results = pd.read_json(raw_results_path, orient="records")

In [ ]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df


raw_results = add_path_metadata(raw_results)
raw_results = format_metric_column(raw_results)
raw_results

In [ ]:
def kl_val(row: pd.Series) -> float:
    # Llama-2-7b-chat-hf-KL-0.0-iter1
    exp_name = row["exp_name"]
    kl_str = exp_name.split("KL-")[-1].split("-")[0]
    return float(kl_str)


logprob_results["kl"] = logprob_results.apply(kl_val, axis=1)
logprob_norm_results["kl"] = logprob_norm_results.apply(kl_val, axis=1)
generative_results["kl"] = generative_results.apply(kl_val, axis=1)

raw_results["kl"] = raw_results.apply(kl_val, axis=1)

In [ ]:
def run_id(df: pd.DataFrame, id_cols: list[str]) -> pd.Series:
    return df[id_cols].astype(str).agg("-".join, axis=1)


run_id_cols = ["model_name", "layer_name", "exp_name"]

logprob_results["run_id"] = run_id(logprob_results, run_id_cols)
logprob_norm_results["run_id"] = run_id(logprob_norm_results, run_id_cols)
generative_results["run_id"] = run_id(generative_results, run_id_cols)

raw_results["run_id"] = run_id(raw_results, run_id_cols)

Process Raw Results

In [ ]:
raw_results["benchmark_metric"] = raw_results["benchmark"] + "/" + raw_results["metric"]

In [ ]:
raw_results.columns.tolist()
pivot_raw_results = raw_results.pivot_table(index=["model_name", "layer_name", "kl"], columns="benchmark_metric", values="value")


In [ ]:
denom = 15 + 50
pivot_raw_results["global/kl_div"] = (15 / denom) * pivot_raw_results["eval-oasst2/kl_div"] + (50 / denom) * pivot_raw_results["eval-tulu-v3/kl_div"]
pivot_raw_results["global/proj_l2_rel"] = (15 / denom) * pivot_raw_results["eval-oasst2/proj_l2_rel"] + (50 / denom) * pivot_raw_results[
    "eval-tulu-v3/proj_l2_rel"
]


In [ ]:
# Get the reference values where kl == 0
ref_values = pivot_raw_results.xs(0.0, level="kl")[["global/kl_div", "global/proj_l2_rel"]]

# Join reference values to the pivot table
# We don't need to manually initialize the _max columns as the join will create them
if "global/kl_div_max" in pivot_raw_results.columns:
    pivot_raw_results = pivot_raw_results.drop(columns=["global/kl_div_max", "global/proj_l2_rel_max"])

pivot_raw_results = pivot_raw_results.join(ref_values, on=["model_name", "layer_name"], rsuffix="_max")

pivot_raw_results = pivot_raw_results[["global/kl_div", "global/kl_div_max", "global/proj_l2_rel", "global/proj_l2_rel_max"]]

Concat logprob with logprob results

In [ ]:
logprob_results["metric"] = "acc"
logprob_norm_results["metric"] = "acc_norm"
generative_results["metric"] = "generative"

In [ ]:
# add logprob_norm_results to logprob_results
logprob_results = pd.concat([logprob_results, logprob_norm_results], ignore_index=True)
logprob_results

In [ ]:
merge_keys = ["model_name", "layer_name", "kl"]

logprob_results = logprob_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

generative_results = generative_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

In [ ]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str = "abs_diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2]).abs()
    return df


# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "abs_diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "abs_diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "abs_diff")

In [ ]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str = "diff") -> pd.DataFrame:
    df[new_col] = df[col1] - df[col2]
    return df


# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "diff")

In [ ]:
threshold = 0.015
generative_results["diff_prob"] = (((generative_results["value"] - generative_results["clean_mean"]).abs()) < threshold).astype(float)

In [ ]:
choice_metric_col = "silence_score"

In [ ]:
logprob_results[choice_metric_col] = logprob_results["lower_tail"] + logprob_results["diff_prob"]
generative_results[choice_metric_col] = generative_results["lower_tail"] + generative_results["diff_prob"]

In [ ]:
logprob_results["diff_prob"].describe()

In [ ]:
# mask = generative_results['abs_diff'] < 0.015
# a=generative_results[mask].groupby(['model_name', 'layer_name', 'kl'])['two_sided_tail'].describe().reset_index()

In [ ]:
mask_logprob = logprob_results["diff"] <= 0.0
logprob_results = logprob_results[mask_logprob]

mask_generative = generative_results["diff"] <= 0.0
generative_results = generative_results[mask_generative]

## Now AGREGRATE

In [ ]:
def agg_results(
    df: pd.DataFrame,
    prob_col: str = "two_sided_tail",
    new_col: str = "two_sided_tail_agg",
    group_by_cols: list[str] | None = None,
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    keep_cols_local = list(keep_cols) if keep_cols is not None else []
    keep_cols_local = list(dict.fromkeys([*keep_cols_local, prob_col]))  # unique, order-preserving

    group_cols = list(group_by_cols) if group_by_cols is not None else []
    selected_cols = list(dict.fromkeys([*group_cols, *keep_cols_local]))

    if group_by_cols is not None:
        idx = df.groupby(group_by_cols, sort=False)[prob_col].idxmin()
        agg_df = df.loc[idx, selected_cols].reset_index(drop=True)
    else:
        idx = df[prob_col].idxmin()
        agg_df = df.loc[[idx], selected_cols].reset_index(drop=True)

    agg_df = agg_df.rename(columns={prob_col: new_col})
    return agg_df

In [ ]:
raw_keep_cols = [
    "global/kl_div",
    "global/kl_div_max",
    "global/proj_l2_rel",
    "global/proj_l2_rel_max",
    "diff_prob",
    "two_sided_tail",
    "lower_tail",
    "upper_tail",
    "silence_score",
]
agg_choice_metric_col = f"agg_{choice_metric_col}"

In [ ]:
abs_diff_threshold = -1e-10

agg_logprobs = agg_results(
    logprob_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

agg_generative = agg_results(
    generative_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

In [ ]:
agg_generative.sort_values("agg_silence_score", ascending=False)

In [ ]:
agg_logprobs["choice_metric"] = agg_logprobs["global/proj_l2_rel"] * agg_logprobs[agg_choice_metric_col]
agg_generative["choice_metric"] = agg_generative["global/proj_l2_rel"] * agg_generative[agg_choice_metric_col]

In [ ]:
metrict_to_keep_part_2 = ["diff_prob", "two_sided_tail", "lower_tail", "upper_tail", "agg_silence_score"]
# create ne dataframe with model_name, layer_name, kl, choice_metric for both logprobs and generative
agg_logprobs_choice = agg_logprobs[["model_name", "layer_name", "kl", "choice_metric"] + metrict_to_keep_part_2]
agg_generative_choice = agg_generative[["model_name", "layer_name", "kl", "choice_metric"] + metrict_to_keep_part_2]

agg_choice = agg_logprobs_choice.merge(
    agg_generative_choice,
    on=["model_name", "layer_name", "kl"],
    suffixes=("_logprob", "_generative"),
)

agg_logprobs_metrics = agg_logprobs[
    ["model_name", "layer_name", "kl", "global/proj_l2_rel", "global/proj_l2_rel_max", "global/kl_div", "global/kl_div_max"]
]

agg_choice = agg_choice.merge(
    agg_logprobs_metrics,
    on=["model_name", "layer_name", "kl"],
    how="left",
)

agg_choice

In [ ]:
# add agg_choice_metric column that is min of choice_metric_logprob and choice_metric_generative
agg_choice["agg_choice_metric"] = agg_choice[[f"choice_metric_logprob", f"choice_metric_generative"]].min(axis=1)

agg_choice["agg_silent_metric"] = agg_choice["agg_choice_metric"] / agg_choice["global/proj_l2_rel"]
# agg_choice['silent_metric_logprob'] = agg_choice['choice_metric_logprob'] / agg_choice['global/proj_l2_rel']
# agg_choice['silent_metric_generative'] = agg_choice['choice_metric_generative'] / agg_choice['global/proj_l2_rel']


agg_choice
# for each group of (model_name, layer_name) take the kl value with the highest value

In [ ]:
# Cleaner: break ties with kl first, then pick max agg_choice_metric per group via idxmax.
_tmp = agg_choice.sort_values(["model_name", "layer_name", "kl"], ascending=[True, True, False])
_idx = _tmp.groupby(["model_name", "layer_name"])["agg_choice_metric"].idxmax()

best_kl_df = (_tmp.loc[_idx, agg_choice.columns.tolist()].sort_values(["model_name", "layer_name"])).reset_index(drop=True)

best_kl_df

In [ ]:
rename_map = {
    # Experiment
    "model_name": "Model",
    "layer_name": "Layer",
    "kl": r"$\lambda$",
    # Experiment Results
    "agg_silent_metric": r"$\mathrm{Sil}$",
    "global/proj_l2_rel": r"$E_{L2}$",
    "global/proj_l2_rel_max": r"$\max E_{L2}$",
    "global/kl_div": r"$\mathcal{L}_{KL}$",
    "global/kl_div_max": r"$\max \mathcal{L}_{KL}$",
    # Logprob Results
    "diff_prob_logprob": r"$\mathrm{Diff}_{\text{logprob}}$",
    "lower_tail_logprob": r"$\mathrm{LTail}_{\text{logprob}}$",
    "agg_silence_score_logprob": r"$\mathrm{Sil}_{\text{logprob}}$",
    # Generative Results
    "diff_prob_generative": r"$\mathrm{Diff}_{\text{generative}}$",
    "lower_tail_generative": r"$\mathrm{LTail}_{\text{generative}}$",
    "agg_silence_score_generative": r"$\mathrm{Sil}_{\text{generative}}$",
}
mask_cols = [k for k in rename_map.keys()]
best_kl_df_renamed = best_kl_df[mask_cols].rename(columns=rename_map)

In [ ]:
best_kl_df_renamed.sort_values(rename_map["agg_silent_metric"], ascending=False)

In [ ]:
best_kl_df_renamed

In [ ]:
def fmt(x, k=4):
    return f"{x:.{k}f}".rstrip("0").rstrip(".")


print(best_kl_df_renamed.to_latex(index=False, float_format=lambda x: fmt(x, k=4)))

In [ ]:
def layer_order(layer_name: str) -> int:
    if ".layers." in layer_name:
        return int(layer_name.split(".")[-1])
    elif "embed_tokens" in layer_name:
        return -1
    else:
        return 1000

In [ ]:
def visualize(
    df: pd.DataFrame,
    res_name: str,
    abs_diff_threshold: float,
    row: str = None,
):
    g = sns.catplot(
        data=df,
        x="two_sided_tail_agg",
        y="layer_name",
        kind="bar",
        col="model_name",
        row=row,
        order=sorted(df["layer_name"].unique(), key=layer_order),
    )
    g.figure.suptitle(f"Aggregated {res_name} two-sided tail values (threshold={abs_diff_threshold})", y=1.02)